# Fine-tune Jigsaw-pretrained ResNet-18, ps32 SCV

This notebook fine-tunes a binary ResNet-18 classifier initialized from the Jigsaw SSL encoder checkpoint. It uses the same ps32 5-cluster spatial cross-validation protocol as the scratch, masked-reconstruction, and contrastive ResNet-18 models.

## 1. Purpose and experiment configuration

This notebook performs supervised fine-tuning only. It does not rerun Jigsaw SSL pretraining, does not modify any SSL pretraining checkpoint, and does not generate susceptibility maps.

Scientific warning: Jigsaw pretraining showed near-random permutation prediction accuracy. Fine-tuning results should be interpreted as an empirical test of a potentially weak spatial-ordering pretext task rather than as evidence that the Jigsaw task itself was successfully solved.

In [1]:
from pathlib import Path
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

model_name = "jigsaw_resnet18"
pretraining = "jigsaw"
patch_size = 32
random_seed = 42
val_fraction = 0.2
batch_size = 16
encoder_learning_rate = 1e-5
head_learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 100
early_stopping_patience = 15
gradient_clip_norm = 5.0
dropout = 0.4

patch_index_csv = PROJECT_ROOT / "data/processed/patches/labeled_patch_index_ps32_common_balanced.csv"
raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
encoder_checkpoint_path = PROJECT_ROOT / "checkpoints/ssl_pretrained/jigsaw/resnet18_jigsaw_ps32_encoder_best.pt"
output_root = PROJECT_ROOT / "outputs/R1_ssl_task_comparison/jigsaw"
figure_root = PROJECT_ROOT / "figures/R1_ssl_task_comparison/jigsaw"
checkpoint_dir = PROJECT_ROOT / "checkpoints/finetuned/jigsaw_resnet18"

reference_metrics = {
    "scratch": {"mean_auc": 0.778842, "std_auc": 0.104754, "worst_fold_auc": 0.624023},
    "masked_recon": {"mean_auc": 0.820912, "std_auc": 0.043011, "worst_fold_auc": 0.786325},
    "contrastive": {"mean_auc": 0.838008, "std_auc": 0.084941, "worst_fold_auc": 0.704142},
}

print("Project root:", PROJECT_ROOT)
print("Patch index:", patch_index_csv)
print("Jigsaw encoder checkpoint:", encoder_checkpoint_path)

Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Patch index: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\patches\labeled_patch_index_ps32_common_balanced.csv
Jigsaw encoder checkpoint: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_encoder_best.pt


## 2. Import packages and project modules

In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.patch_dataset import list_raster_files, audit_raster_alignment
from src.models_resnet18 import create_resnet18_binary_classifier, load_ssl_encoder_weights_into_classifier
from src.plotting import set_publication_plot_style
from src.train_finetune import run_pretrained_resnet18_scv_experiment
from src.utils import count_trainable_parameters, ensure_dir, set_global_seed

## 3. Set random seeds and device

In [3]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    print("torch.version.cuda:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available; training will run on CPU.")

Using device: cuda
CUDA GPU: NVIDIA GeForce RTX 4090
torch.version.cuda: 11.8


## 4. Set publication-quality plotting style

In [4]:
plot_font_family = set_publication_plot_style(font_family="Arial", font_size=10)
print("Plotting font family:", plot_font_family)
print("Plotting font size:", 10)

Plotting font family: Arial
Plotting font size: 10


## 5. Load patch index, cleaned rasters, and Jigsaw pretrained encoder checkpoint

In [5]:
patch_index = pd.read_csv(patch_index_csv).reset_index(drop=True)
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=-9999)
jigsaw_checkpoint = torch.load(encoder_checkpoint_path, map_location="cpu")
expected_checkpoint_keys = {
    "encoder_state_dict", "epoch", "val_loss", "val_acc_top1", "val_acc_top5",
    "config", "channel_means", "channel_stds", "permutation_bank_path",
}
missing_checkpoint_keys = expected_checkpoint_keys - set(jigsaw_checkpoint.keys())
if missing_checkpoint_keys:
    raise KeyError(f"Jigsaw checkpoint missing keys: {sorted(missing_checkpoint_keys)}")
jigsaw_pretext_val_loss = float(jigsaw_checkpoint["val_loss"])
jigsaw_pretext_val_acc_top1 = float(jigsaw_checkpoint["val_acc_top1"])
jigsaw_pretext_val_acc_top5 = float(jigsaw_checkpoint["val_acc_top5"])
print("Patch index rows:", len(patch_index))
print("Raster channels:", len(raster_files))
print("Raster shape:", (raster_audit.height, raster_audit.width))
print("Jigsaw checkpoint epoch:", jigsaw_checkpoint.get("epoch"))
print("Jigsaw pretext val_loss:", jigsaw_pretext_val_loss)
print("Jigsaw pretext val_acc_top1:", jigsaw_pretext_val_acc_top1)
print("Jigsaw pretext val_acc_top5:", jigsaw_pretext_val_acc_top5)
print("Jigsaw encoder state keys:", len(jigsaw_checkpoint["encoder_state_dict"]))

Patch index rows: 344
Raster channels: 14
Raster shape: (10071, 9596)
Jigsaw checkpoint epoch: 12
Jigsaw pretext val_loss: 4.605131553649902
Jigsaw pretext val_acc_top1: 0.012000000059604644
Jigsaw pretext val_acc_top5: 0.04650000005960465
Jigsaw encoder state keys: 120


## 6. Verify input data quality

In [6]:
required_columns = ["sample_id", "x", "y", "label", "source", "cluster_id", "valid_patch", "nodata_ratio"]
missing_columns = [column for column in required_columns if column not in patch_index.columns]
if missing_columns:
    raise ValueError(f"Patch index missing required columns: {missing_columns}")
if not patch_index["valid_patch"].astype(bool).all():
    raise ValueError("Patch index contains invalid patches.")
if not np.isclose(patch_index["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
    raise ValueError("Patch index contains nonzero nodata_ratio rows.")
print("total sample count:", len(patch_index))
print("label counts:")
print(patch_index["label"].value_counts().sort_index().to_string())
print("cluster_id x label table:")
print(pd.crosstab(patch_index["cluster_id"], patch_index["label"]).to_string())
print("patch_size:", patch_size)
print("number of raster channels:", len(raster_files))
print("dataset path:", patch_index_csv)
print("Jigsaw encoder checkpoint path:", encoder_checkpoint_path)
print("device:", device)
print("batch size:", batch_size)
print("encoder learning rate:", encoder_learning_rate)
print("head learning rate:", head_learning_rate)
print("max epochs:", max_epochs)
print("early stopping patience:", early_stopping_patience)
print("plotting font family and font size:", plot_font_family, 10)

total sample count: 344
label counts:
label
0    172
1    172
cluster_id x label table:
label        0   1
cluster_id        
0           32  32
1           39  39
2           53  53
3           14  14
4           34  34
patch_size: 32
number of raster channels: 14
dataset path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\patches\labeled_patch_index_ps32_common_balanced.csv
Jigsaw encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_encoder_best.pt
device: cuda
batch size: 16
encoder learning rate: 1e-05
head learning rate: 0.0001
max epochs: 100
early stopping patience: 15
plotting font family and font size: Arial 10


## 7. Define modified ResNet-18 classifier and load Jigsaw SSL encoder weights

In [7]:
preview_model = create_resnet18_binary_classifier(in_channels=14, dropout=dropout, small_patch_stem=True, pretrained=False)
preview_load_info = load_ssl_encoder_weights_into_classifier(preview_model, encoder_checkpoint_path, strict_encoder=True)
loaded_preview_keys = len(preview_load_info["loaded_keys"])
if loaded_preview_keys < 110:
    raise RuntimeError(f"Too few Jigsaw encoder keys loaded: {loaded_preview_keys}")
print("checkpoint epoch:", jigsaw_checkpoint.get("epoch"))
print("checkpoint val_loss:", jigsaw_pretext_val_loss)
print("checkpoint val_acc_top1:", jigsaw_pretext_val_acc_top1)
print("checkpoint val_acc_top5:", jigsaw_pretext_val_acc_top5)
print("preview loaded pretrained encoder keys:", loaded_preview_keys)
print("model parameter count:", count_trainable_parameters(preview_model))
del preview_model

Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']
checkpoint epoch: 12
checkpoint val_loss: 4.605131553649902
checkpoint val_acc_top1: 0.012000000059604644
checkpoint val_acc_top5: 0.04650000005960465
preview loaded pretrained encoder keys: 120
model parameter count: 11175681


## 8. Define SCV fold loop

Fold loop follows cluster_id 0..4 as held-out test clusters, matching scratch, masked-reconstruction, and contrastive experiments.

## 9. Compute fold-specific channel normalization from training data only

For fair comparison, supervised fine-tuning normalization is computed from training samples within each fold only. The Jigsaw SSL checkpoint means/stds are not used for supervised fine-tuning.

## 10. Fine-tune fold-specific Jigsaw-pretrained ResNet-18 models

In [8]:
scientific_note = "Jigsaw pretraining showed near-random pretext accuracy; therefore, downstream fine-tuning results should be interpreted as an ablation of a weak spatial-ordering pretext task."
results = run_pretrained_resnet18_scv_experiment(
    project_root=PROJECT_ROOT,
    model_name=model_name,
    pretraining=pretraining,
    patch_index_csv=patch_index_csv,
    raster_dir=raster_dir,
    encoder_checkpoint_path=encoder_checkpoint_path,
    output_root=output_root,
    figure_root=figure_root,
    checkpoint_dir=checkpoint_dir,
    patch_size=patch_size,
    random_seed=random_seed,
    val_fraction=val_fraction,
    batch_size=batch_size,
    encoder_learning_rate=encoder_learning_rate,
    head_learning_rate=head_learning_rate,
    weight_decay=weight_decay,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience,
    gradient_clip_norm=gradient_clip_norm,
    dropout=dropout,
    reference_metrics=reference_metrics,
    expected_checkpoint_keys=expected_checkpoint_keys,
    checkpoint_extra_values={
        "jigsaw_pretext_val_loss": jigsaw_pretext_val_loss,
        "jigsaw_pretext_val_acc_top1": jigsaw_pretext_val_acc_top1,
        "jigsaw_pretext_val_acc_top5": jigsaw_pretext_val_acc_top5,
    },
    scientific_note=scientific_note,
    device=device,
    plot_figures=True,
)
fold_metrics = results["fold_metrics"]
summary_df = results["summary_metrics"]
paths = results["paths"]

Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']

Fold 0: held-out test cluster 0
train clusters: [1, 2, 3, 4]
validation size: 56
test cluster: 0
train label counts:
label
0    112
1    112
val label counts:
label
0    28
1    28
test label counts:
label
0    32
1    32


channel means and stds shape: (14,) (14,)
Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']


model parameter count: 11175681
number of trainable parameters: 11175681
number of loaded pretrained encoder keys: 120


device sanity model: cuda:0
device sanity X: cuda:0
device sanity y: cuda:0


Fold 0 test AUC: 0.843750; best threshold: 0.086064

Fold 1: held-out test cluster 1
train clusters: [0, 2, 3, 4]
validation size: 54
test cluster: 1
train label counts:
label
0    106
1    106
val label counts:
label
0    27
1    27
test label counts:
label
0    39
1    39


...

Fold 3 test AUC: 0.821429; best threshold: 0.595718

Fold 4: held-out test cluster 4
train clusters: [0, 1, 2, 3]
validation size: 56
test cluster: 4
train label counts:
label
0    110
1    110
val label counts:
label
0    28
1    28
test label counts:
label
0    34
1    34


channel means and stds shape: (14,) (14,)
Loaded SSL encoder keys: 120
Missing keys after load: 2
Unexpected/skipped keys: 0
Expected classifier-head missing keys: ['fc.1.bias', 'fc.1.weight']
model parameter count: 11175681
number of trainable parameters: 11175681
number of loaded pretrained encoder keys: 120


device sanity model: cuda:0
device sanity X: cuda:0
device sanity y: cuda:0


Fold 4 test AUC: 0.605536; best threshold: 0.195184


Jigsaw pretraining showed near-random pretext accuracy; therefore, downstream fine-tuning results should be interpreted as an ablation of a weak spatial-ordering pretext task.


## 11. Evaluate fold-specific test performance

In [9]:
print(fold_metrics[["fold", "auc", "pr_auc", "accuracy_05", "f1_05", "best_threshold_f1", "f1_best_f1", "best_epoch"]].to_string(index=False))

 fold      auc   pr_auc  accuracy_05    f1_05  best_threshold_f1  f1_best_f1  best_epoch
    0 0.843750 0.788072     0.843750 0.857143           0.086064    0.714286          29
    1 0.690335 0.696427     0.628205 0.491228           0.840441    0.384615          55
    2 0.651477 0.603884     0.613208 0.661157           0.059208    0.670968          20
    3 0.821429 0.826576     0.750000 0.774194           0.595718    0.774194          21
    4 0.605536 0.660490     0.514706 0.421053           0.195184    0.626506          15


## 12. Save predictions, metrics, logs, and checkpoints

In [10]:
prediction_columns = ["sample_id", "x", "y", "label", "source", "cluster_id", "fold", "y_true", "y_logit", "y_prob", "y_pred_05", "y_pred_best_f1", "split"]
metric_columns = ["model", "pretraining", "patch_size", "fold", "n_train", "n_val", "n_test", "n_test_pos", "n_test_neg", "auc", "pr_auc", "accuracy_05", "precision_05", "recall_05", "f1_05", "best_threshold_f1", "accuracy_best_f1", "precision_best_f1", "recall_best_f1", "f1_best_f1", "tn", "fp", "fn", "tp", "best_epoch", "best_val_auc", "best_val_loss"]
if list(fold_metrics.columns) != metric_columns:
    raise ValueError("Fold metrics schema mismatch.")
for fold_id in range(5):
    pred_path = output_root / "predictions" / f"jigsaw_resnet18_ps32_fold{fold_id}_predictions.csv"
    log_path = output_root / "training_logs" / f"jigsaw_resnet18_ps32_fold{fold_id}_training_log.csv"
    ckpt_path = checkpoint_dir / f"jigsaw_resnet18_ps32_fold{fold_id}_best.pt"
    pred = pd.read_csv(pred_path)
    if list(pred.columns) != prediction_columns:
        raise ValueError(f"Prediction columns mismatch for fold {fold_id}.")
    log = pd.read_csv(log_path)
    for column in ["epoch", "train_loss", "val_loss", "train_auc", "val_auc", "train_f1", "val_f1", "learning_rate", "encoder_learning_rate", "head_learning_rate"]:
        if column not in log.columns:
            raise ValueError(f"Training log fold {fold_id} missing {column}.")
    ckpt = torch.load(ckpt_path, map_location="cpu")
    for key in ["model_state_dict", "optimizer_state_dict", "epoch", "best_val_auc", "best_val_loss", "fold", "patch_size", "model_name", "pretraining_type", "encoder_checkpoint_path", "jigsaw_pretext_val_loss", "jigsaw_pretext_val_acc_top1", "jigsaw_pretext_val_acc_top5", "channel_means", "channel_stds", "config"]:
        if key not in ckpt:
            raise KeyError(f"Checkpoint fold {fold_id} missing key {key}.")
print("Prediction files, training logs, fold metrics, summary metrics, and checkpoints passed schema checks.")
print("Fold metrics:", paths["fold_metrics"])
print("Summary metrics:", paths["summary_metrics"])

Prediction files, training logs, fold metrics, summary metrics, and checkpoints passed schema checks.
Fold metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\jigsaw\metrics\jigsaw_resnet18_ps32_fold_metrics.csv
Summary metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\jigsaw\metrics\jigsaw_resnet18_ps32_summary_metrics.csv


## 13. Plot ROC, PR, and training curves using standardized figure style

In [11]:
figure_paths = [
    figure_root / "roc_curves/jigsaw_resnet18_ps32_roc_all_folds.png",
    figure_root / "roc_curves/jigsaw_resnet18_ps32_roc_all_folds.pdf",
    figure_root / "pr_curves/jigsaw_resnet18_ps32_pr_all_folds.png",
    figure_root / "pr_curves/jigsaw_resnet18_ps32_pr_all_folds.pdf",
    figure_root / "training_curves/jigsaw_resnet18_ps32_training_curves.png",
    figure_root / "training_curves/jigsaw_resnet18_ps32_training_curves.pdf",
]
for path in figure_paths:
    if not path.exists():
        raise FileNotFoundError(path)
    print(path)

D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\roc_curves\jigsaw_resnet18_ps32_roc_all_folds.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\roc_curves\jigsaw_resnet18_ps32_roc_all_folds.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\pr_curves\jigsaw_resnet18_ps32_pr_all_folds.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\pr_curves\jigsaw_resnet18_ps32_pr_all_folds.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\training_curves\jigsaw_resnet18_ps32_training_curves.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\jigsaw\training_curves\jigsaw_resnet18_ps32_training_curves.pdf


## 14. Compare summary metrics against scratch, masked-recon, and contrastive baselines

In [12]:
summary = summary_df.iloc[0].to_dict()
print(summary_df.T.to_string(header=False))

mean_auc                                0.722506
std_auc                                 0.105176
mean_pr_auc                              0.71509
std_pr_auc                              0.091449
mean_accuracy_05                        0.669974
std_accuracy_05                         0.128138
mean_precision_05                       0.676622
std_precision_05                        0.117925
mean_recall_05                          0.652255
std_recall_05                           0.278139
mean_f1_05                              0.640955
std_f1_05                               0.184169
mean_accuracy_best_f1                   0.605546
std_accuracy_best_f1                    0.090529
mean_precision_best_f1                   0.61849
std_precision_best_f1                   0.113604
mean_recall_best_f1                     0.759378
std_recall_best_f1                       0.29303
mean_f1_best_f1                         0.634114
std_f1_best_f1                          0.149763
model               

## 15. Print final summary and scientific interpretation

In [13]:
mean_std_table = pd.DataFrame({
    "metric": ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"],
    "mean": [summary[f"mean_{m}"] for m in ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]],
    "std": [summary[f"std_{m}"] for m in ["auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]],
})
print("Fold-wise metrics table:")
print(fold_metrics[["fold", "auc", "pr_auc", "accuracy_05", "f1_05", "accuracy_best_f1", "f1_best_f1"]].to_string(index=False))
print("\nMean +/- std metrics:")
print(mean_std_table.to_string(index=False))
print("\nWorst fold by AUC:", summary["worst_fold"], summary["worst_fold_auc"])
print("Best fold by AUC:", summary["best_fold"], summary["best_fold_auc"])
print("Scratch baseline mean_auc/std_auc:", reference_metrics["scratch"]["mean_auc"], reference_metrics["scratch"]["std_auc"])
print("Masked-recon mean_auc/std_auc:", reference_metrics["masked_recon"]["mean_auc"], reference_metrics["masked_recon"]["std_auc"])
print("Contrastive mean_auc/std_auc:", reference_metrics["contrastive"]["mean_auc"], reference_metrics["contrastive"]["std_auc"])
print("Jigsaw mean_auc/std_auc:", summary["mean_auc"], summary["std_auc"])
for key in ["delta_mean_auc_vs_scratch", "delta_std_auc_vs_scratch", "delta_worst_auc_vs_scratch", "delta_mean_auc_vs_masked_recon", "delta_std_auc_vs_masked_recon", "delta_worst_auc_vs_masked_recon", "delta_mean_auc_vs_contrastive", "delta_std_auc_vs_contrastive", "delta_worst_auc_vs_contrastive"]:
    print(key + ":", summary[key])
print("\nSaved output locations:")
for key, value in paths.items():
    print(key + ":", value)
print("\nScientific note:", scientific_note)

Fold-wise metrics table:
 fold      auc   pr_auc  accuracy_05    f1_05  accuracy_best_f1  f1_best_f1
    0 0.843750 0.788072     0.843750 0.857143          0.625000    0.714286
    1 0.690335 0.696427     0.628205 0.491228          0.589744    0.384615
    2 0.651477 0.603884     0.613208 0.661157          0.518868    0.670968
    3 0.821429 0.826576     0.750000 0.774194          0.750000    0.774194
    4 0.605536 0.660490     0.514706 0.421053          0.544118    0.626506

Mean +/- std metrics:
          metric     mean      std
             auc 0.722506 0.105176
          pr_auc 0.715090 0.091449
     accuracy_05 0.669974 0.128138
           f1_05 0.640955 0.184169
accuracy_best_f1 0.605546 0.090529
      f1_best_f1 0.634114 0.149763

Worst fold by AUC: 4 0.6055363321799307
Best fold by AUC: 0 0.84375
Scratch baseline mean_auc/std_auc: 0.778842 0.104754
Masked-recon mean_auc/std_auc: 0.820912 0.043011
Contrastive mean_auc/std_auc: 0.838008 0.084941
Jigsaw mean_auc/std_auc: 0.72250